<h1><center>AG News Classification</center></h1>

AG is a collection of more than 1 million news articles. News articles have been gathered from more than 2000 news sources by ComeToMyHead in more than 1 year of activity. ComeToMyHead is an academic news search engine which has been running since July, 2004. The dataset is provided by the academic comunity for research purposes in data mining (clustering, classification, etc), information retrieval (ranking, search, etc), xml, data compression, data streaming, and any other non-commercial activity.

The AG's news topic classification dataset is constructed by choosing 4 largest classes from the original corpus. Each class contains 30,000 training samples and 1,900 testing samples. The total number of training samples is 120,000 and testing 7,600.

In [1]:
# Import libraries
from pathlib import Path
import torch
import torch.nn as nn

from src.data.data_loader import create_dataloaders
from src.model.transformer import build_transformer
from src.model.transformer import Transformer
from src.train.training import train_model
from src.utils.utils import get_device
from nltk.tokenize import word_tokenize

from src.utils.constants import PADDING_ID, UNKNOWN_ID
from src.utils.constants import PADDING_VALUE, UNKNOWN_VALUE

/opt/anaconda3/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/opt/anaconda3/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c1017RegisterOperatorsD1Ev
  Referenced from: <CFED5F8E-EC3F-36FD-AAA3-2C6C7F8D3DD9> /opt/anaconda3/lib/python3.11/site-packages/torchvision/image.so
  Expected in:     <CDAC6E34-8608-3E70-8B2F-32BCD38E90FB> /opt/anaconda3/lib/python3.11/site-packages/torch/lib/libtorch_cpu.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [2]:
# Initialize model and training parameters

train_file = 'data/train/train.csv'
val_file = 'data/val/val.csv'

# Size of embedding vector
d_model = 64
# Number of words in a vocabulary
vocab_size = 30000
# Max sequence length for input words/tokens
seq_len = 100
# Dropout rate
dropout = 0.1
# number of encoder blocks
num_layers = 1
# number of attention heads
num_heads = 8
# Number of hidden nodes for feed-forward layer
d_ff = 4*64

# Number of epochs
epochs = 5
# Batch size for training
batch_size = 128
# Number of classes
num_classes = 4

In [3]:
# Get a device to use for training/inference
device = get_device()

# Create training and validation data loaders
train_dataloader, val_dataloader, word_to_id = create_dataloaders(train_file, val_file, batch_size, seq_len,
                                                                   vocab_size)

In [4]:
# Create encoder only transformer model
encoder_only_transformer_model = build_transformer(d_model, vocab_size, seq_len, dropout,
                                                   num_layers, num_heads, d_ff, num_classes).to(device)

print(encoder_only_transformer_model)

Transformer(
  (embed): InputEmbedding(
    (embedding): Embedding(30000, 64)
  )
  (pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Encoder(
    (layers): ModuleList(
      (0): EncoderBlock(
        (self_attention): MultiHeadAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (query_linear_layer): Linear(in_features=64, out_features=64, bias=True)
          (key_linear_layer): Linear(in_features=64, out_features=64, bias=True)
          (value_linear_layer): Linear(in_features=64, out_features=64, bias=True)
          (output_linear_layer): Linear(in_features=64, out_features=64, bias=True)
        )
        (feed_forward): FeedForward(
          (linear_1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear_2): Linear(in_features=256, out_features=64, bias=True)
        )
        (dropout): Dropout(p=0.1, inplace=False)
        (norm): LayerNorm((64,), e

In [6]:
# Create optimizer and loss function
optimizer = torch.optim.Adam(encoder_only_transformer_model.parameters())
loss_fn = nn.CrossEntropyLoss()

# Train model
train_model(epochs, num_classes, encoder_only_transformer_model, train_dataloader, val_dataloader,
            loss_fn, optimizer, device)

Epoch 1, Train Loss 0.048310380429029465, Train Accuracy 0.9821928143501282, Validation Loss 0.3696085214614868, Validation Accuracy 0.918836772441864
Epoch 2, Train Loss 0.028784865513443947, Train Accuracy 0.9894806146621704, Validation Loss 0.42624807357788086, Validation Accuracy 0.9156249761581421
Epoch 3, Train Loss 0.023488840088248253, Train Accuracy 0.9912880063056946, Validation Loss 0.5056792497634888, Validation Accuracy 0.9140190482139587
Epoch 4, Train Loss 0.02079736813902855, Train Accuracy 0.9923707246780396, Validation Loss 0.48174676299095154, Validation Accuracy 0.9132378101348877
Epoch 5, Train Loss 0.016717806458473206, Train Accuracy 0.9940364956855774, Validation Loss 0.58421391248703, Validation Accuracy 0.910937488079071


In [7]:
# Save model

# Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Create model save path
MODEL_NAME = "06_news_classification.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

In [8]:
# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=encoder_only_transformer_model.state_dict(), f=MODEL_SAVE_PATH)

Saving model to: models/06_news_classification.pth
